# Tesla Autopilot Safety Records — EDA

This notebook uses the project pipeline to validate and clean the provided incident CSV, create trend/fatality/model summaries, and save an interactive HTML report. These are reported records, not exposure-normalized crash rates or evidence of Autopilot causation.

In [ ]:
import sys
from pathlib import Path

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "Module-5" / "GENAI").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Run this notebook from the repository or one of its subfolders.")
sys.path.insert(0, str(ROOT / "Module-5" / "GENAI" / "projects"))
from autonomous_driving import load_safety_data, build_safety_report

csv_path = ROOT / "datasets" / "GEN AI" / "Capstone 1" / "Part 2" / "Tesla - Deaths.csv"
records, quality = load_safety_data(csv_path)
print(f"Clean incidents: {len(records)}")
print(f"Date range: {records.date.min().date()} to {records.date.max().date()}")
quality

In [ ]:
records[["date", "country", "state", "model", "deaths", "driver", "occupant",
         "other_vehicle", "cyclists_pedestrians", "autopilot_claimed",
         "verified_autopilot_deaths"]].head(10)

In [ ]:
annual = records.groupby("year", as_index=False).agg(
    events=("case_id", "nunique"),
    reported_deaths=("deaths", "sum"),
    verified_autopilot_events=("verified_autopilot_event", "sum"),
    autopilot_claimed_events=("autopilot_claimed_event", "sum"),
)
annual

In [ ]:
annual_summary = build_safety_report(
    records,
    quality,
    str(ROOT / "Module-5" / "GENAI" / "projects" / "autonomous_driving.py"),
)
print("Interactive report saved under Module-5/GENAI/projects/reports/")
annual_summary